In [1]:
%cd ..

/Users/ext-elias.melas/Documents/Gitcode/opensearch-testing


In [2]:
from opensearchpy import OpenSearch

# Create Mock data (100k accounts)

In [5]:
from src.make_index_mock import main
main()

Connected to OpenSearch 3.0.0 at http://localhost:9200
Index 'game_accounts' exists – deleting …
✅  Created index 'game_accounts'
Indexing 100,000 docs (batch 5000) …
✅ Successfully indexed 100000 documents
🎉  Done – 'game_accounts' now holds 100,000 documents


## Search Mock data

In [9]:
from src.utils import search_utils
from dotenv import load_dotenv
load_dotenv()

client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200, 'scheme': 'http'}],  # Explicitly use HTTP
    http_auth=('admin', 'admin'),  # Default credentials for GitHub Actions OpenSearch
    use_ssl=False,
    verify_certs=False,
    ssl_show_warn=False,
)

In [10]:
from src.utils.search_utils import GameAccountSearcher, QueryType

# Basic search
searcher = GameAccountSearcher(client)
results = searcher.add_query("player_tag", "ABC12", QueryType.TERM).search()

# Complex search with multiple conditions
searcher = GameAccountSearcher(client)
results = (
    searcher
    .add_query("alliance_name", " knight", QueryType.MATCH, fuzziness="AUTO")
    .add_query("subscription_status", "premium", QueryType.TERM, context="filter")
    .add_range_query("last_login", gte="now-30d/d")
    .search(size=20, from_=0)
)

results

{'took': 16,
 'timed_out': False,
 '_shards': {'total': 3, 'successful': 3, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 5271, 'relation': 'eq'},
  'max_score': 1.6086862,
  'hits': [{'_index': 'game_accounts',
    '_id': 'f75b2b5d-e66d-4d27-b2b1-24e74ecc88ee',
    '_score': 1.6086862,
    '_source': {'account_id': 'ACC0000004',
     'player_id': '78e8dae6-7a63-4a2f-9289-3f5fef0b2b23',
     'player_tag': '#0209031',
     'avatar': 'Seamless tertiary hardware',
     'alliance_name': 'Shadow Knights',
     'level': 90,
     'email': 'bradleyashlee@example.org',
     'phone_number': '001-778-905-1194x974',
     'country': 'TT',
     'device_id': '72ce79d2-96de-417e-9bf1-ead399ae95b6',
     'registration_date': '2024-11-12T01:25:09.902201',
     'last_login': '2029-05-27T01:25:09.902201',
     'ip_address': '25.220.169.7',
     'subscription_status': 'premium',
     'account_status': 'banned',
     'preferred_language': 'fr',
     'date_of_birth': '2004-08-30'}},
   {'_index': 

In [15]:
no_of_hits = results['hits']['total']['value']
print(f'no_of_hits: {no_of_hits}')

no_of_hits: 5202


In [16]:
# Multi-field search
searcher = GameAccountSearcher(client)
results = searcher.add_query(
    "search_term",
    "dragon",
    QueryType.MULTI_MATCH,
    fields=["alliance_name", "avatar"]
).search()

results

{'took': 15,
 'timed_out': False,
 '_shards': {'total': 3, 'successful': 3, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 9900, 'relation': 'eq'},
  'max_score': 1.0568901,
  'hits': [{'_index': 'game_accounts',
    '_id': '4875eb10-1d09-48ae-9ada-2f9fc4ebb19f',
    '_score': 1.0568901,
    '_source': {'account_id': 'ACC0000000',
     'player_id': 'f3a9cd7b-3dd7-4f1f-8c02-4f94babe08b1',
     'player_tag': '#1967825',
     'avatar': 'Diverse fresh-thinking adapter',
     'alliance_name': 'Dragon Warriors',
     'level': 95,
     'email': 'james08@example.com',
     'phone_number': '+1-336-935-6602',
     'country': 'KZ',
     'device_id': '29f6efc7-42d0-4085-8215-c6d6c3c220e9',
     'registration_date': '2025-04-17T23:14:06.680326',
     'last_login': '2028-11-16T23:14:06.680326',
     'ip_address': '104.71.223.149',
     'subscription_status': 'premium',
     'account_status': 'active',
     'preferred_language': 'es',
     'date_of_birth': '1998-04-23'}},
   {'_index': 'gam